# Lab 8: Deep Observability — Tracing, Metrics & Monitoring in Azure AI Foundry
---
**SmartClaims AI Agent** | Microsoft Foundry Agent Service (v2.x)

## Objective
Build a **production-grade observability pipeline** for your AI agents. You'll learn:
- **Distributed tracing** with OpenTelemetry and Azure Monitor (Application Insights)
- **Custom span attributes** to capture agent-specific metadata (tokens, tools, latency)
- **Metrics collection** — track token usage, response times, error rates
- **Log correlation** — link traces, metrics, and logs with a single `trace_id`
- **KQL queries** — query your agent telemetry in Application Insights
- **Alerting patterns** — set up rules for latency spikes, failures, and cost anomalies

## Why Observability Matters for AI Agents
Unlike traditional APIs, AI agents are **non-deterministic** and **multi-step**. A single user query can trigger:
- Multiple LLM calls (reasoning loops)
- Tool/function executions (file search, code interpreter, web search)
- Conversation state management

Without observability, you're flying blind:

| Question | Without Observability | With Observability |
|----------|----------------------|-------------------|
| Why was the response slow? | "I don't know" | "Code Interpreter took 8.2s" |
| How much are we spending? | "Check the bill next month" | "42K tokens/hr, $12.60/day" |
| Is the agent hallucinating? | "Users are complaining" | "Grounding score dropped below 0.7 at 2pm" |
| Which tool fails most? | "Let me check logs" | "file_search: 3.2% error rate (timeout)" |

## Prerequisites
- Completed **Lab 0** (connection verified)
- Azure Application Insights resource (or we'll use console exporters as fallback)
- `pip install opentelemetry-sdk azure-monitor-opentelemetry azure-core-tracing-opentelemetry`

## The Three Pillars of Observability

```
                    ┌─────────────────────────────────────┐
                    │       Azure Application Insights     │
                    │    (Unified Observability Backend)    │
                    └──────┬──────────┬──────────┬────────┘
                           │          │          │
                    ┌──────┴───┐ ┌────┴────┐ ┌───┴──────┐
                    │  Traces  │ │ Metrics │ │   Logs   │
                    │ (Spans)  │ │(Counters│ │(Structured│
                    │          │ │ Gauges) │ │ Events)  │
                    └──────┬───┘ └────┬────┘ └───┬──────┘
                           │          │          │
                    ┌──────┴──────────┴──────────┴────────┐
                    │        OpenTelemetry SDK              │
                    │   TracerProvider + MeterProvider      │
                    │   + LoggerProvider                    │
                    └──────┬──────────┬──────────┬────────┘
                           │          │          │
               ┌───────────┴───┐  ┌───┴───┐  ┌──┴──────────┐
               │ Agent Call    │  │ Token  │  │ Error Logs  │
               │ create_agent │  │ Count  │  │ Tool Fails  │
               │ run_thread   │  │ Latency│  │ Auth Errors │
               └───────────────┘  └───────┘  └─────────────┘
```

| Pillar | What It Captures | Azure Table |
|--------|-----------------|-------------|
| **Traces** | End-to-end request flow across agent calls, tool executions | `dependencies`, `requests` |
| **Metrics** | Counters (tokens used), histograms (latency), gauges (active sessions) | `customMetrics` |
| **Logs** | Structured events — errors, warnings, agent decisions | `traces` (yes, confusing name!) |

## Step 1: Configure OpenTelemetry with Azure Monitor

We'll set up OpenTelemetry to capture distributed traces and metrics from every agent call. The `ConsoleSpanExporter` prints traces locally; for production, switch to Azure Monitor.

> **Two setup paths:**
> - **Path A (Production):** `configure_azure_monitor()` — sends to Application Insights
> - **Path B (Local dev):** Console exporters — prints to notebook output
>
> We'll configure **both** so you can see traces locally AND in Azure.

In [1]:
import sys, os, time, json
from datetime import datetime, timezone

# MUST be set BEFORE importing any Azure SDK
os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"

sys.path.insert(0, os.path.join(os.getcwd(), ".."))

# ─── OpenTelemetry Imports ───────────────────────────────
from opentelemetry import trace, metrics
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import (
    SimpleSpanProcessor, ConsoleSpanExporter, BatchSpanProcessor,
)
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import (
    ConsoleMetricExporter, PeriodicExportingMetricReader,
)
from opentelemetry.sdk.resources import Resource

# ─── Azure SDK Imports ───────────────────────────────────
from azure.ai.projects.models import PromptAgentDefinition
from azure.core.exceptions import HttpResponseError, ClientAuthenticationError

# ─── Shared Config ───────────────────────────────────────
from utils.config import get_clients, MODEL, print_header, print_step

# ─── Resource: identifies this service in all telemetry ──
resource = Resource.create({
    "service.name": "smartclaims-agent",
    "service.version": "2.0.0",
    "deployment.environment": "lab",
})

print("✅ Core imports loaded")
print(f"   GenAI tracing: {os.environ.get('AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING')}")

✅ Core imports loaded
   GenAI tracing: true


In [2]:
# ─── Setup: Azure Monitor (Production) OR Console (Local Dev) ───

# Option A: Azure Monitor — uncomment if you have an App Insights resource
# from azure.monitor.opentelemetry import configure_azure_monitor
# APPINSIGHTS_CONN_STRING = os.environ.get("APPLICATIONINSIGHTS_CONNECTION_STRING")
# if APPINSIGHTS_CONN_STRING:
#     configure_azure_monitor(
#         connection_string=APPINSIGHTS_CONN_STRING,
#         resource=resource,
#     )
#     print("✅ Azure Monitor configured — telemetry flows to Application Insights")

# Option B: Console exporters (always active for lab visibility)
# Traces → console
trace_provider = TracerProvider(resource=resource)
trace_provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(trace_provider)

# Metrics → console (export every 10 seconds)
metric_reader = PeriodicExportingMetricReader(
    ConsoleMetricExporter(),
    export_interval_millis=10_000,
)
meter_provider = MeterProvider(resource=resource, metric_readers=[metric_reader])
metrics.set_meter_provider(meter_provider)

# Get handles
tracer = trace.get_tracer("smartclaims.observability", "1.0.0")
meter  = metrics.get_meter("smartclaims.observability", "1.0.0")

print("✅ TracerProvider configured (console exporter)")
print("✅ MeterProvider configured (console exporter, 10s interval)")
print()
print("💡 For production, uncomment Option A above and set:")
print("   APPLICATIONINSIGHTS_CONNECTION_STRING=InstrumentationKey=xxx;...")

✅ TracerProvider configured (console exporter)
✅ MeterProvider configured (console exporter, 10s interval)

💡 For production, uncomment Option A above and set:
   APPLICATIONINSIGHTS_CONNECTION_STRING=InstrumentationKey=xxx;...


In [3]:
# ─── Create Clients ──────────────────────────────────────
project_client, openai_client = get_clients()
print("✅ AIProjectClient created")
print("✅ OpenAI client created")

✅ AIProjectClient created
✅ OpenAI client created


## Step 2: Define Custom Metrics

Metrics give you **aggregated numbers** — counters, histograms, gauges. Unlike traces (which capture individual requests), metrics answer questions like:
- "How many tokens did we use in the last hour?"
- "What's the p95 response latency?"
- "How many agent errors occurred today?"

We'll define four key metrics:

| Metric | Type | What It Tracks |
|--------|------|----------------|
| `agent.request.count` | Counter | Total agent calls (by status) |
| `agent.request.duration` | Histogram | Response time distribution |
| `agent.tokens.used` | Counter | Input + output tokens consumed |
| `agent.active_sessions` | UpDownCounter | Currently active conversations |

In [4]:
# ─── Custom Metrics ──────────────────────────────────────

# Counter: total number of agent requests
request_counter = meter.create_counter(
    name="agent.request.count",
    description="Total number of agent requests",
    unit="requests",
)

# Histogram: response time distribution (seconds)
latency_histogram = meter.create_histogram(
    name="agent.request.duration",
    description="Agent response time in seconds",
    unit="s",
)

# Counter: total tokens consumed (input + output)
token_counter = meter.create_counter(
    name="agent.tokens.used",
    description="Total tokens consumed by agent calls",
    unit="tokens",
)

# UpDownCounter: currently active sessions (goes up and down)
active_sessions = meter.create_up_down_counter(
    name="agent.active_sessions",
    description="Number of currently active agent sessions",
    unit="sessions",
)

print("✅ Custom metrics defined:")
print("   - agent.request.count    (Counter)")
print("   - agent.request.duration (Histogram)")
print("   - agent.tokens.used      (Counter)")
print("   - agent.active_sessions  (UpDownCounter)")

✅ Custom metrics defined:
   - agent.request.count    (Counter)
   - agent.request.duration (Histogram)
   - agent.tokens.used      (Counter)
   - agent.active_sessions  (UpDownCounter)


## Step 3: Instrumented Agent Call Wrapper

This is the heart of observability — an `observed_call()` function that wraps every agent interaction with:

1. **A trace span** — captures the full lifecycle of the call
2. **Span attributes** — agent name, version, model, user input length
3. **Metric recordings** — latency, token count, success/failure
4. **Error capture** — exceptions are recorded on the span with full stack traces

```
┌─ Span: agent.call ─────────────────────────────────────────┐
│  Attributes:                                                │
│    agent.name = "smartclaims-observability"                  │
│    agent.version = 2                                        │
│    model = "gpt-4o-mini"                                    │
│    input.length = 45                                        │
│    output.length = 128                                      │
│    tokens.input = 62                                        │
│    tokens.output = 89                                       │
│    status = "ok"                                            │
│                                                             │
│  Events:                                                    │
│    [t=0ms]   agent.call.start                               │
│    [t=1.2s]  agent.call.complete                            │
│                                                             │
│  Metrics Recorded:                                          │
│    agent.request.count     += 1  {status: "success"}        │
│    agent.request.duration  += 1.2s                          │
│    agent.tokens.used       += 151                           │
└─────────────────────────────────────────────────────────────┘
```

In [5]:
from opentelemetry.trace import StatusCode

def observed_call(openai_client, agent, user_input, conversation_id=None):
    """
    Instrumented agent call — wraps every interaction with
    traces, metrics, and structured error handling.
    """
    # Common attributes for both span and metrics
    common_attrs = {
        "agent.name": agent.name,
        "agent.version": str(agent.version),
        "model": MODEL,
    }

    with tracer.start_as_current_span("agent.call", attributes=common_attrs) as span:
        span.set_attribute("input.length", len(user_input))
        span.add_event("agent.call.start", {"user_input_preview": user_input[:100]})
        
        # Track active session
        active_sessions.add(1, common_attrs)
        start_time = time.time()

        try:
            # Build request
            kwargs = {
                "extra_body": {
                    "agent_reference": {
                        "name": agent.name,
                        "version": agent.version,
                        "type": "agent_reference",
                    }
                },
                "input": user_input,
            }
            if conversation_id:
                kwargs["conversation"] = conversation_id

            # Execute agent call
            response = openai_client.responses.create(**kwargs)
            duration = time.time() - start_time

            # Extract token usage from response
            input_tokens = getattr(response.usage, "input_tokens", 0) if response.usage else 0
            output_tokens = getattr(response.usage, "output_tokens", 0) if response.usage else 0
            total_tokens = input_tokens + output_tokens

            # Set span attributes
            span.set_attribute("output.length", len(response.output_text))
            span.set_attribute("tokens.input", input_tokens)
            span.set_attribute("tokens.output", output_tokens)
            span.set_attribute("tokens.total", total_tokens)
            span.set_attribute("duration_seconds", round(duration, 3))
            span.set_attribute("status", "success")
            span.set_status(StatusCode.OK)
            span.add_event("agent.call.complete", {
                "duration_ms": int(duration * 1000),
                "total_tokens": total_tokens,
            })

            # Record metrics
            request_counter.add(1, {**common_attrs, "status": "success"})
            latency_histogram.record(duration, common_attrs)
            if total_tokens > 0:
                token_counter.add(total_tokens, common_attrs)

            return {
                "ok": True,
                "text": response.output_text,
                "tokens": {"input": input_tokens, "output": output_tokens, "total": total_tokens},
                "duration": round(duration, 3),
                "trace_id": format(span.get_span_context().trace_id, "032x"),
            }

        except ClientAuthenticationError as e:
            duration = time.time() - start_time
            span.set_status(StatusCode.ERROR, "Authentication failed")
            span.record_exception(e)
            span.set_attribute("status", "auth_error")
            request_counter.add(1, {**common_attrs, "status": "auth_error"})
            latency_histogram.record(duration, common_attrs)
            return {"ok": False, "error": "Auth failed. Run: az login", "error_type": "auth"}

        except HttpResponseError as e:
            duration = time.time() - start_time
            error_type = "rate_limited" if e.status_code == 429 else f"http_{e.status_code}"
            span.set_status(StatusCode.ERROR, str(e))
            span.record_exception(e)
            span.set_attribute("status", error_type)
            span.set_attribute("http.status_code", e.status_code)
            request_counter.add(1, {**common_attrs, "status": error_type})
            latency_histogram.record(duration, common_attrs)
            return {"ok": False, "error": f"HTTP {e.status_code}: {e.message}", "error_type": error_type}

        except Exception as e:
            duration = time.time() - start_time
            span.set_status(StatusCode.ERROR, str(e))
            span.record_exception(e)
            span.set_attribute("status", "exception")
            request_counter.add(1, {**common_attrs, "status": "exception"})
            latency_histogram.record(duration, common_attrs)
            return {"ok": False, "error": str(e), "error_type": "exception"}

        finally:
            active_sessions.add(-1, common_attrs)

print("✅ observed_call() defined — every agent call is now instrumented")

✅ observed_call() defined — every agent call is now instrumented


## Step 4: Create an Agent & Test Instrumented Calls

Let's create an agent and run a few queries through `observed_call()` to generate telemetry data. Watch the console output — you'll see OpenTelemetry spans printed for each call.

In [6]:
# ─── Create Agent ────────────────────────────────────────
agent = project_client.agents.create_version(
    agent_name="smartclaims-observability",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are SmartClaims, an insurance claims assistant. "
            "Be concise and helpful. Always cite policy sections when relevant."
        ),
    ),
)
print(f"✅ Agent created: {agent.name} v{agent.version}")

✅ Agent created: smartclaims-observability v1


In [7]:
# ─── Test Queries: Generate Telemetry ────────────────────
test_queries = [
    "What is the deductible for auto collision claims?",
    "How long do I have to file a homeowner's claim after an incident?",
    "Explain the difference between comprehensive and collision coverage.",
]

results = []
for i, query in enumerate(test_queries, 1): #what is 1 for? 
    # The '1' in enumerate(test_queries, 1) means that the counting of 'i' will start from 1 instead of the default 0.
    print(f"\n{'─'*50}")
    print(f"  Query {i}: {query[:60]}...")
    print(f"{'─'*50}")
    
    result = observed_call(openai_client, agent, query)
    results.append(result)
    
    if result["ok"]:
        print(f"  ✅ Response: {result['text'][:120]}...")
        print(f"  ⏱️  Duration: {result['duration']}s")
        print(f"  🔢 Tokens: {result['tokens']}")
        print(f"  🔍 Trace ID: {result['trace_id']}")
    else:
        print(f"  ❌ Error: {result['error']}")

print(f"\n{'='*50}")
print(f"  All {len(test_queries)} queries complete — check span output above!")
print(f"{'='*50}")


──────────────────────────────────────────────────
  Query 1: What is the deductible for auto collision claims?...
──────────────────────────────────────────────────
{
    "name": "agent.call",
    "context": {
        "trace_id": "0x4648eef121e229e5284c6ae6b77529f7",
        "span_id": "0xb402bde9adb0a4ab",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-03-29T16:23:34.106526Z",
    "end_time": "2026-03-29T16:23:38.055465Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "agent.name": "smartclaims-observability",
        "agent.version": "1",
        "model": "gpt-4o-mini",
        "input.length": 49,
        "output.length": 394,
        "tokens.input": 42,
        "tokens.output": 73,
        "tokens.total": 115,
        "duration_seconds": 3.949,
        "status": "success"
    },
    "events": [
        {
            "name": "agent.call.start",
            "timestamp": "2026-03-29T1

## Step 5: Nested Spans — Tracing Multi-Step Operations

Real agent workflows involve **multiple steps**: validation, agent call, post-processing. Nested spans show the full breakdown:

```
┌─ Span: claims.workflow ──────────────────────────────────┐
│                                                           │
│  ┌─ Span: input.validation ─────────┐                    │
│  │  duration: 0.001s                 │                    │
│  └───────────────────────────────────┘                    │
│                                                           │
│  ┌─ Span: agent.call ──────────────────────────────────┐ │
│  │  duration: 1.4s                                      │ │
│  │  tokens: 145                                         │ │
│  └──────────────────────────────────────────────────────┘ │
│                                                           │
│  ┌─ Span: response.postprocess ─────┐                    │
│  │  duration: 0.002s                 │                    │
│  └───────────────────────────────────┘                    │
│                                                           │
│  total duration: 1.403s                                   │
└───────────────────────────────────────────────────────────┘
```

This is critical for diagnosing latency — is it the model, the network, or your code?

In [8]:
def claims_workflow(openai_client, agent, user_input):
    """
    Full claims workflow with nested spans for each phase.
    Parent span captures the entire operation; child spans
    break down validation, agent call, and post-processing.
    """
    with tracer.start_as_current_span("claims.workflow") as parent_span:
        parent_span.set_attribute("workflow.type", "claims_query")
        parent_span.set_attribute("user_input", user_input[:200])
        workflow_start = time.time()

        # ── Phase 1: Input Validation ──
        with tracer.start_as_current_span("input.validation") as val_span:
            # Simulate validation checks
            is_valid = len(user_input.strip()) > 0 and len(user_input) < 5000
            val_span.set_attribute("input.length", len(user_input))
            val_span.set_attribute("input.valid", is_valid)
            if not is_valid:
                val_span.set_status(StatusCode.ERROR, "Invalid input")
                return {"ok": False, "error": "Input validation failed"}

        # ── Phase 2: Agent Call (uses observed_call internally) ──
        with tracer.start_as_current_span("agent.inference") as call_span:
            kwargs = {
                "extra_body": {
                    "agent_reference": {
                        "name": agent.name,
                        "version": agent.version,
                        "type": "agent_reference",
                    }
                },
                "input": user_input,
            }
            call_start = time.time()
            response = openai_client.responses.create(**kwargs)
            call_duration = time.time() - call_start
            
            call_span.set_attribute("duration_seconds", round(call_duration, 3))
            input_tokens = getattr(response.usage, "input_tokens", 0) if response.usage else 0
            output_tokens = getattr(response.usage, "output_tokens", 0) if response.usage else 0
            call_span.set_attribute("tokens.total", input_tokens + output_tokens)

        # ── Phase 3: Post-Processing ──
        with tracer.start_as_current_span("response.postprocess") as post_span:
            output_text = response.output_text
            # Simulate post-processing: check for policy citations
            has_citation = any(kw in output_text.lower() for kw in ["section", "policy", "article", "clause"])
            post_span.set_attribute("has_citation", has_citation)
            post_span.set_attribute("response_length", len(output_text))

        # ── Record on parent span ──
        total_duration = time.time() - workflow_start
        parent_span.set_attribute("workflow.duration", round(total_duration, 3))
        parent_span.set_attribute("workflow.has_citation", has_citation)
        parent_span.set_status(StatusCode.OK)

        # Record metrics
        request_counter.add(1, {"agent.name": agent.name, "status": "success"})
        latency_histogram.record(total_duration, {"agent.name": agent.name})

        return {
            "ok": True,
            "text": output_text,
            "duration": round(total_duration, 3),
            "has_citation": has_citation,
            "trace_id": format(parent_span.get_span_context().trace_id, "032x"),
        }

print("✅ claims_workflow() defined with nested spans")

✅ claims_workflow() defined with nested spans


In [9]:
# ─── Run the Workflow ────────────────────────────────────
print_step("Running claims_workflow with nested spans")

result = claims_workflow(
    openai_client, agent,
    "What documentation do I need to submit for a water damage claim on my home?"
)

if result["ok"]:
    print(f"\n✅ Workflow complete!")
    print(f"   Response: {result['text'][:150]}...")
    print(f"   Duration: {result['duration']}s")
    print(f"   Has citation: {result['has_citation']}")
    print(f"   Trace ID: {result['trace_id']}")
    print(f"\n📊 Check the console output above — you'll see THREE nested spans:")
    print(f"   1. input.validation (child)")
    print(f"   2. agent.inference (child)")
    print(f"   3. response.postprocess (child)")
    print(f"   4. claims.workflow (parent — ends last)")
else:
    print(f"❌ {result['error']}")


──────────────────────────────────────────────────
  Running claims_workflow with nested spans
──────────────────────────────────────────────────
{
    "name": "input.validation",
    "context": {
        "trace_id": "0xf01477d99934586e3c74714e540736a5",
        "span_id": "0xfdb8483cca2d6df6",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x8bcceff7b509cdb3",
    "start_time": "2026-03-29T16:23:42.657564Z",
    "end_time": "2026-03-29T16:23:42.657564Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "input.length": 75,
        "input.valid": true
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.40.0",
            "service.name": "smartclaims-agent",
            "service.version": "2.0.0",
            "deployment.environment": "lab"
       

## Step 6: Observability Dashboard — Visualize Metrics Locally

Before you have Application Insights set up, let's build a **local dashboard** using matplotlib to visualize the telemetry we've collected. This mirrors what you'd see in Azure Monitor dashboards.

We'll simulate a load test (multiple queries) and plot:
1. **Response latency** per query
2. **Token usage** per query
3. **Cumulative cost estimate**

In [10]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for Jupyter

# ─── Load Test: 6 diverse queries ───────────────────────
load_test_queries = [
    "What is the claims filing deadline?",
    "Do I need a police report for a theft claim?",
    "What's covered under comprehensive auto insurance?",
    "How do I appeal a denied claim?",
    "What is subrogation and how does it affect my claim?",
    "Can I get a rental car while my vehicle is being repaired?",
]

print("🔄 Running load test with 6 queries...\n")
load_results = []
for i, q in enumerate(load_test_queries, 1):
    r = observed_call(openai_client, agent, q)
    load_results.append(r)
    status = "✅" if r["ok"] else "❌"
    duration = r.get("duration", 0)
    tokens = r.get("tokens", {}).get("total", 0)
    print(f"  {status} Q{i}: {duration}s | {tokens} tokens | {q[:40]}...")

print(f"\n✅ Load test complete — {len(load_results)} queries executed")

🔄 Running load test with 6 queries...

{
    "resource_metrics": [
        {
            "resource": {
                "attributes": {
                    "telemetry.sdk.language": "python",
                    "telemetry.sdk.name": "opentelemetry",
                    "telemetry.sdk.version": "1.40.0",
                    "service.name": "smartclaims-agent",
                    "service.version": "2.0.0",
                    "deployment.environment": "lab"
                },
                "schema_url": ""
            },
            "scope_metrics": [
                {
                    "scope": {
                        "name": "opentelemetry-sdk",
                        "version": null,
                        "schema_url": "",
                        "attributes": null
                    },
                    "metrics": [
                        {
                            "name": "otel.sdk.span.started",
                            "description": "The number of created spa

In [11]:
# ─── Plot Observability Dashboard ────────────────────────
successful = [r for r in load_results if r["ok"]]
durations = [r["duration"] for r in successful]
total_tokens = [r["tokens"]["total"] for r in successful]
input_tokens = [r["tokens"]["input"] for r in successful]
output_tokens = [r["tokens"]["output"] for r in successful]
labels = [f"Q{i+1}" for i in range(len(successful))]

# Cost estimate: gpt-4o-mini pricing (approx)
COST_PER_1K_INPUT  = 0.00015   # $0.15 per 1M input tokens
COST_PER_1K_OUTPUT = 0.0006    # $0.60 per 1M output tokens
costs = [(inp * COST_PER_1K_INPUT + out * COST_PER_1K_OUTPUT) / 1000
         for inp, out in zip(input_tokens, output_tokens)]
cumulative_cost = [sum(costs[:i+1]) for i in range(len(costs))]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("SmartClaims Agent — Observability Dashboard", fontsize=14, fontweight="bold")

# Plot 1: Latency
ax1 = axes[0]
bars1 = ax1.bar(labels, durations, color="#4A90D9", edgecolor="white")
ax1.axhline(y=sum(durations)/len(durations), color="red", linestyle="--", label=f"Avg: {sum(durations)/len(durations):.2f}s")
ax1.set_title("Response Latency (seconds)")
ax1.set_ylabel("Seconds")
ax1.legend()
for bar, val in zip(bars1, durations):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{val:.2f}s", ha="center", va="bottom", fontsize=9)

# Plot 2: Token Usage (stacked)
ax2 = axes[1]
ax2.bar(labels, input_tokens, color="#50C878", edgecolor="white", label="Input")
ax2.bar(labels, output_tokens, bottom=input_tokens, color="#FF6B6B", edgecolor="white", label="Output")
ax2.set_title("Token Usage (Input vs Output)")
ax2.set_ylabel("Tokens")
ax2.legend()

# Plot 3: Cumulative Cost
ax3 = axes[2]
ax3.plot(labels, cumulative_cost, marker="o", color="#9B59B6", linewidth=2, markersize=8)
ax3.fill_between(labels, cumulative_cost, alpha=0.2, color="#9B59B6")
ax3.set_title("Cumulative Cost Estimate ($)")
ax3.set_ylabel("USD")
for i, cost in enumerate(cumulative_cost):
    ax3.annotate(f"${cost:.5f}", (labels[i], cost), textcoords="offset points",
                 xytext=(0, 10), ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("../outputs/observability_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n📊 Dashboard saved to outputs/observability_dashboard.png")


📊 Dashboard saved to outputs/observability_dashboard.png


C:\Users\losts\AppData\Local\Temp\ipykernel_32884\1374718541.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 7: KQL Queries for Application Insights

When you're using Azure Monitor (Option A from Step 1), your telemetry lands in Application Insights tables. Here are the **essential KQL queries** for monitoring your agent in production.

> **How to run these:** Azure Portal → Application Insights → Logs (Analytics)

### 7.1 — Agent Call Latency (p50, p95, p99)
```kusto
dependencies
| where name == "agent.call"
| where timestamp > ago(24h)
| summarize 
    p50 = percentile(duration, 50),
    p95 = percentile(duration, 95),
    p99 = percentile(duration, 99),
    avg_duration = avg(duration),
    total_calls = count()
    by bin(timestamp, 1h)
| order by timestamp desc
```

### 7.2 — Token Usage Over Time
```kusto
dependencies
| where name == "agent.call"
| where timestamp > ago(24h)
| extend input_tokens = toint(customDimensions["tokens.input"]),
         output_tokens = toint(customDimensions["tokens.output"])
| summarize 
    total_input = sum(input_tokens),
    total_output = sum(output_tokens),
    avg_tokens_per_call = avg(input_tokens + output_tokens)
    by bin(timestamp, 1h)
| order by timestamp desc
```

### 7.3 — Error Rate by Type
```kusto
dependencies
| where name == "agent.call"
| where timestamp > ago(24h)
| extend status = tostring(customDimensions["status"])
| summarize count() by status, bin(timestamp, 1h)
| render timechart
```

### 7.4 — Slowest Agent Calls (Investigate Outliers)
```kusto
dependencies
| where name == "agent.call"
| where timestamp > ago(24h)
| where duration > 5000  // calls slower than 5 seconds
| project timestamp, duration, 
    agent_name = customDimensions["agent.name"],
    tokens = customDimensions["tokens.total"],
    input_preview = customDimensions["user_input_preview"]
| order by duration desc
| take 20
```

### 7.5 — End-to-End Workflow Trace
```kusto
// Find all spans for a specific trace ID
union dependencies, requests
| where operation_Id == "YOUR_TRACE_ID_HERE"
| project timestamp, name, duration, success,
    customDimensions
| order by timestamp asc
```

### 7.6 — Cost Estimation Dashboard
```kusto
dependencies
| where name == "agent.call"
| where timestamp > ago(7d)
| extend input_tokens = toint(customDimensions["tokens.input"]),
         output_tokens = toint(customDimensions["tokens.output"])
| extend cost_usd = (input_tokens * 0.00000015) + (output_tokens * 0.0000006)
| summarize 
    daily_cost = sum(cost_usd),
    total_calls = count(),
    total_tokens = sum(input_tokens + output_tokens)
    by bin(timestamp, 1d)
| order by timestamp desc
```

## Step 8: Alerting Patterns

Set up Azure Monitor alerts to catch issues before users do.

### Recommended Alert Rules

| Alert | Condition | Severity | Action |
|-------|-----------|----------|--------|
| **High Latency** | p95 latency > 10s for 5 min | Sev 2 (Warning) | Notify Slack/Teams |
| **Error Spike** | Error rate > 5% in 15 min | Sev 1 (Error) | Page on-call |
| **Token Budget** | Daily tokens > 500K | Sev 3 (Info) | Email team lead |
| **Auth Failures** | Any auth_error in 5 min | Sev 1 (Error) | Page on-call + check credentials |
| **Rate Limiting** | >3 rate_limited errors in 5 min | Sev 2 (Warning) | Scale up or throttle |

### Example: Create Alert via Azure CLI
```bash
# High latency alert (p95 > 10 seconds)
az monitor metrics alert create \
  --name "SmartClaims-HighLatency" \
  --resource-group "your-rg" \
  --scopes "/subscriptions/.../applicationInsights/your-app" \
  --condition "avg dependencies/duration > 10000 where dependency/name == 'agent.call'" \
  --window-size 5m \
  --evaluation-frequency 1m \
  --severity 2 \
  --action-group "your-action-group"
```

### Example: Create Alert via Terraform
```hcl
resource "azurerm_monitor_metric_alert" "agent_latency" {
  name                = "smartclaims-high-latency"
  resource_group_name = azurerm_resource_group.main.name
  scopes              = [azurerm_application_insights.main.id]
  severity            = 2
  frequency           = "PT1M"
  window_size         = "PT5M"

  criteria {
    metric_namespace = "microsoft.insights/components"
    metric_name      = "dependencies/duration"
    aggregation      = "Average"
    operator         = "GreaterThan"
    threshold        = 10000

    dimension {
      name     = "dependency/name"
      operator = "Include"
      values   = ["agent.call"]
    }
  }

  action {
    action_group_id = azurerm_monitor_action_group.oncall.id
  }
}
```

## Step 9: Foundry Built-in Tracing (Portal View)

Azure AI Foundry provides **built-in tracing** that you can view directly in the portal — no code changes needed beyond the environment variable we already set.

### How to View Traces in the Portal

1. Go to **Azure AI Foundry** → your project
2. Click **Tracing** in the left sidebar
3. You'll see all agent interactions with:
   - Request/response details
   - Token usage per call
   - Latency breakdown
   - Tool/function call details

### What Foundry Auto-Captures

| Data | Auto-Captured? | Details |
|------|---------------|---------|
| Agent create/delete | ✅ | Name, version, model, instructions |
| Responses.create | ✅ | Input, output, tokens, latency |
| Function calls | ✅ | Function name, arguments, results |
| File search | ✅ | Query, results, vector store ID |
| Code interpreter | ✅ | Code executed, output, files generated |
| Errors | ✅ | HTTP status, error message, stack trace |

### Connecting Foundry Tracing to Application Insights

```python
# In your .env file, add:
# APPLICATIONINSIGHTS_CONNECTION_STRING=InstrumentationKey=...

# Then in code:
from azure.monitor.opentelemetry import configure_azure_monitor

configure_azure_monitor(
    connection_string=os.environ["APPLICATIONINSIGHTS_CONNECTION_STRING"],
)

# Now ALL Foundry SDK spans flow to Application Insights
# AND you can add your own custom spans on top
```

This gives you the best of both worlds:
- **Foundry portal** for quick debugging during development
- **Application Insights** for production monitoring, alerting, and long-term analysis

## Step 10: Telemetry Summary Report

Let's generate a summary of all the telemetry we've collected in this lab — this is what you'd review in a daily standup or weekly ops review.

In [12]:
# ─── Telemetry Summary ───────────────────────────────────
all_results = results + load_results  # from Steps 4 and 6
successful = [r for r in all_results if r["ok"]]
failed = [r for r in all_results if not r["ok"]]

all_durations = [r["duration"] for r in successful]
all_tokens = [r["tokens"]["total"] for r in successful]
all_input_tokens = [r["tokens"]["input"] for r in successful]
all_output_tokens = [r["tokens"]["output"] for r in successful]

total_cost = sum(
    (inp * COST_PER_1K_INPUT + out * COST_PER_1K_OUTPUT) / 1000
    for inp, out in zip(all_input_tokens, all_output_tokens)
)

print("=" * 60)
print("  OBSERVABILITY SUMMARY REPORT")
print(f"  Generated: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")
print("=" * 60)
print()
print(f"  Total Requests:     {len(all_results)}")
print(f"  Successful:         {len(successful)} ({len(successful)/len(all_results)*100:.0f}%)")
print(f"  Failed:             {len(failed)} ({len(failed)/len(all_results)*100:.0f}%)")
print()
print("  LATENCY")
print(f"  ├─ Average:         {sum(all_durations)/len(all_durations):.3f}s")
print(f"  ├─ Min:             {min(all_durations):.3f}s")
print(f"  ├─ Max:             {max(all_durations):.3f}s")
sorted_d = sorted(all_durations)
p95_idx = int(len(sorted_d) * 0.95)
print(f"  └─ p95:             {sorted_d[min(p95_idx, len(sorted_d)-1)]:.3f}s")
print()
print("  TOKEN USAGE")
print(f"  ├─ Total Input:     {sum(all_input_tokens):,}")
print(f"  ├─ Total Output:    {sum(all_output_tokens):,}")
print(f"  ├─ Total:           {sum(all_tokens):,}")
print(f"  └─ Avg per call:    {sum(all_tokens)/len(all_tokens):,.0f}")
print()
print("  COST ESTIMATE (gpt-4o-mini)")
print(f"  ├─ This session:    ${total_cost:.6f}")
print(f"  ├─ Projected/hour:  ${total_cost * 60:.4f} (at current rate)")
print(f"  └─ Projected/day:   ${total_cost * 1440:.4f}")
print()
print("=" * 60)

  OBSERVABILITY SUMMARY REPORT
  Generated: 2026-03-29 16:24 UTC

  Total Requests:     9
  Successful:         9 (100%)
  Failed:             0 (0%)

  LATENCY
  ├─ Average:         2.383s
  ├─ Min:             1.700s
  ├─ Max:             3.949s
  └─ p95:             3.949s

  TOKEN USAGE
  ├─ Total Input:     389
  ├─ Total Output:    1,061
  ├─ Total:           1,450
  └─ Avg per call:    161

  COST ESTIMATE (gpt-4o-mini)
  ├─ This session:    $0.000695
  ├─ Projected/hour:  $0.0417 (at current rate)
  └─ Projected/day:   $1.0007



## Step 11: Clean Up

In [ ]:
# ─── Clean Up ────────────────────────────────────────────
try:
    project_client.agents.delete_agent("smartclaims-observability")
    print("✅ Deleted: smartclaims-observability")
except Exception:
    print("⏭️  Skip: smartclaims-observability (already deleted)")

# Flush any pending telemetry
trace_provider.force_flush()
meter_provider.force_flush()
print("✅ Telemetry flushed")

⏭️  Skip: smartclaims-observability (already deleted)
{
    "resource_metrics": [
        {
            "resource": {
                "attributes": {
                    "telemetry.sdk.language": "python",
                    "telemetry.sdk.name": "opentelemetry",
                    "telemetry.sdk.version": "1.40.0",
                    "service.name": "smartclaims-agent",
                    "service.version": "2.0.0",
                    "deployment.environment": "lab"
                },
                "schema_url": ""
            },
            "scope_metrics": [
                {
                    "scope": {
                        "name": "opentelemetry-sdk",
                        "version": null,
                        "schema_url": "",
                        "attributes": null
                    },
                    "metrics": [
                        {
                            "name": "otel.sdk.span.started",
                            "description": "The number

{
    "resource_metrics": [
        {
            "resource": {
                "attributes": {
                    "telemetry.sdk.language": "python",
                    "telemetry.sdk.name": "opentelemetry",
                    "telemetry.sdk.version": "1.40.0",
                    "service.name": "smartclaims-agent",
                    "service.version": "2.0.0",
                    "deployment.environment": "lab"
                },
                "schema_url": ""
            },
            "scope_metrics": [
                {
                    "scope": {
                        "name": "opentelemetry-sdk",
                        "version": null,
                        "schema_url": "",
                        "attributes": null
                    },
                    "metrics": [
                        {
                            "name": "otel.sdk.span.started",
                            "description": "The number of created spans.",
                            "unit

## Lab 8 Complete!

### Key Takeaways

| Concept | What You Learned |
|---------|------------------|
| **Three Pillars** | Traces, Metrics, Logs — and how they work together |
| **OpenTelemetry Setup** | TracerProvider + MeterProvider + Resource identification |
| **Custom Metrics** | Counters, Histograms, UpDownCounters for agent-specific KPIs |
| **Instrumented Calls** | `observed_call()` — every agent interaction gets traces + metrics |
| **Nested Spans** | Break down multi-step workflows to find latency bottlenecks |
| **Local Dashboard** | Matplotlib visualization of latency, tokens, and cost |
| **KQL Queries** | 6 production-ready queries for Application Insights |
| **Alerting** | Azure Monitor alerts for latency, errors, cost, and auth failures |
| **Foundry Tracing** | Built-in portal view + Application Insights integration |

### Production Checklist

- [ ] Set `APPLICATIONINSIGHTS_CONNECTION_STRING` in your `.env`
- [ ] Replace `ConsoleSpanExporter` with `configure_azure_monitor()`
- [ ] Pin the Application Insights KQL queries as dashboard tiles
- [ ] Configure alert rules (latency, error rate, token budget)
- [ ] Set up Azure Workbook for weekly ops review
- [ ] Enable Foundry portal tracing for development debugging

### Architecture: Full Observability Stack

```
┌─────────────────────────────────────────────────────────────┐
│                    Your Application                          │
│  ┌─────────────┐  ┌──────────────┐  ┌───────────────────┐  │
│  │  FastAPI /   │  │  observed_   │  │  claims_workflow  │  │
│  │  Streamlit   │──│  call()      │──│  (nested spans)   │  │
│  └─────────────┘  └──────────────┘  └───────────────────┘  │
└──────────────────────────┬──────────────────────────────────┘
                           │ OpenTelemetry SDK
                           │ (auto + manual instrumentation)
                           ▼
              ┌────────────────────────────┐
              │   Azure Monitor Exporter    │
              │   (traces + metrics + logs) │
              └─────────────┬──────────────┘
                            ▼
              ┌────────────────────────────┐
              │   Application Insights      │
              │  ┌────────┐ ┌────────────┐ │
              │  │  KQL   │ │ Dashboards │ │
              │  │ Queries │ │ & Workbooks│ │
              │  └────────┘ └────────────┘ │
              │  ┌────────┐ ┌────────────┐ │
              │  │ Alerts │ │  Smart     │ │
              │  │ & Rules│ │ Detection  │ │
              │  └────────┘ └────────────┘ │
              └────────────────────────────┘
```

---
**Next →** [Lab 9: Streaming](lab9_streaming.ipynb) — Stream agent responses in real-time